In [16]:
from ssb import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m", "max_dpd_12m"],            
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m", "payment_ratio_avg_6m", "payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m", "utilization_avg_6m", "utilization_avg_12m", "utilization_max_12m"],
# }

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000, 1000, 500],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="Stress_Level",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=5,
    max_segments=10,
    enable_diversity=False,
    max_feature_reuse=2,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None, #business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/Datasets/student-lifestyle-and-stress-dataset.csv").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-05 03:10:15,290 | INFO     | [universal_data_loader.py:75] | Detecting local file format handler for extension: '.csv'...
2026-07-05 03:10:15,323 | INFO     | [ssb.py:338] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-05 03:10:15,324 | INFO     | [ssb.py:359] | Iteration 1 | Remaining Volume: 25,500 | Base Rate: 29.99%
2026-07-05 03:10:19,074 | INFO     | [ssb.py:459] | Feature Usage Tracker Update -> 'Study_Hours' used count = 1
2026-07-05 03:10:19,074 | INFO     | [ssb.py:459] | Feature Usage Tracker Update -> 'Exam_Pressure' used count = 1
2026-07-05 03:10:19,075 | INFO     | [ssb.py:474] | Segment 1 Captured (Size Floor: 500 | Lift Floor: 1.5): (Study_Hours >= 4.14 AND Study_Hours < 5.38) AND Exam_Pressure >= 8.50
2026-07-05 03:10:19,078 | INFO     | [ssb.py:359] | Iteration 2 | Remaining Volume: 24,980 | Base Rate: 28.85%
2026-07-05 03:10:19,665 | INFO     | [ssb.py:459] | Feature Usage Tracker Update -> 'Exam_Pressure' used count = 2
2026-07-05 03:10:19,6

In [17]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate   |        lift        |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
|   0.0   |   20576.0   |     4439.0    | 21.573678071539657 | 29.98823529411765  | 80.69019607843137 | 0.719404721883433  |
|   1.0   |    520.0    |     441.0     |  84.8076923076923  | 29.98823529411765  |  2.03921568627451 | 2.8280321091227325 |
|   2.0   |    2124.0   |     1741.0    | 81.96798493408663  | 29.98823529411765  | 8.329411764705883 | 2.7333380617486713 |
|   3.0   |    1196.0   |     637.0     | 53.26086956521739  | 29.98823529411765  | 4.690196078431373 | 1.7760588124925374 |
|   4.0   |    1084.0   |     389.0     | 35.88560885608856  | 29.98823529411765  | 4.250980392156863 | 1.196656238826021  |


In [18]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Study_Hours=[4.14, 5.38) & Exam_Pressure=[8.50, inf)
SQL Filter: (Study_Hours >= 4.14 AND Study_Hours < 5.38) AND Exam_Pressure >= 8.50
--------------------------------------------------
Segment ID: 2
Raw Rule:   Exam_Pressure=[8.50, inf)
SQL Filter: Exam_Pressure >= 8.50
--------------------------------------------------
Segment ID: 3
Raw Rule:   Study_Hours=[8.44, inf)
SQL Filter: Study_Hours >= 8.44
--------------------------------------------------
Segment ID: 4
Raw Rule:   Sleep_Hours=[2.04, 3.96)
SQL Filter: (Sleep_Hours >= 2.04 AND Sleep_Hours < 3.96)
--------------------------------------------------


In [31]:
pd.DataFrame(final_eval)

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,20576,4439.0,21.573678,29.988235,80.690196,0.719405
1,1,520,441.0,84.807692,29.988235,2.039216,2.828032
2,2,2124,1741.0,81.967985,29.988235,8.329412,2.733338
3,3,1196,637.0,53.260870,29.988235,4.690196,1.776059
4,4,1084,389.0,35.885609,29.988235,4.250980,1.196656


In [19]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN (Study_Hours >= 4.14 AND Study_Hours < 5.38) AND Exam_Pressure >= 8.50
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN Exam_Pressure >= 8.50
            THEN 1 ELSE 0 END AS seg_2,
            CASE WHEN Study_Hours >= 8.44
            THEN 1 ELSE 0 END AS seg_3,
            CASE WHEN (Sleep_Hours >= 2.04 AND Sleep_Hours < 3.96)
            THEN 1 ELSE 0 END AS seg_4,
            FROM predicted
""").df()
conn.close()

In [20]:
scorer = StrategicSegmentScore(
    target_col="Stress_Level",
    primary_key="index",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4"],
)

In [21]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-05 03:12:02,129 | INFO     | [ssb.py:546] | Initializing DuckDB scorecard engine...
2026-07-05 03:12:02,169 | INFO     | [ssb.py:581] | Computing harmonic scorecard weights...
2026-07-05 03:12:02,169 | INFO     | [ssb.py:618] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-05 03:12:02,172 | INFO     | [ssb.py:635] | Dataset Zero-Inflation Rate: 70.01%
2026-07-05 03:12:02,172 | INFO     | [ssb.py:641] | Normal Distribution (<80%). Deciling across full dataset...
2026-07-05 03:12:02,173 | INFO     | [ssb.py:650] | Calibrating deciles across 25,500 target customers...


{'model_metadata': {'total_training_population': 25500,
  'active_scored_population': 25500,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2999},
 'segment_weights': {'seg_1': {'weight': 31,
   'lift': 2.828,
   'response_rate': 0.8481,
   'capture_rate': 0.0577},
  'seg_2': {'weight': 117,
   'lift': 2.752,
   'response_rate': 0.8253,
   'capture_rate': 0.2853},
  'seg_3': {'weight': 33,
   'lift': 1.9159,
   'response_rate': 0.5745,
   'capture_rate': 0.1003},
  'seg_4': {'weight': 21,
   'lift': 1.5182,
   'response_rate': 0.4553,
   'capture_rate': 0.0812}},
 'decile_min_thresholds': {'1': 117,
  '2': 0,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [22]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-05 03:12:53,362 | INFO     | [1327506462.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/Scores/scorecard_model_test1.json...


In [23]:
model_artifact.get("decile_min_thresholds")

{'1': 117,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [24]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 31
Segment: seg_2 | Weight: 117
Segment: seg_3 | Weight: 33
Segment: seg_4 | Weight: 21


In [25]:
model_artifact.get("decile_min_thresholds")

{'1': 117,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [40]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 31 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 117 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 33 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 21 ELSE 0 END AS seg_4_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=117 THEN 1
                    WHEN total_weight >= 0 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [38]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Stress_Level) AS events, 
                    (SUM(Stress_Level)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [39]:
scored

,decile_band,count,events,response_rate
0,1,2644,2182.0,82.526475
1,2,22856,5465.0,23.910571
